## 0. Proof of lifeRun this first. It prints immediately, so a blank log means the run hasnot started — not that it is stuck. It also tells you whether Internetis on, which is off by default on Kaggle and breaks every install.

In [ ]:
# Immediate proof of life. Kaggle's first log lines are debugger noise; this is# the first thing that is actually yours, so it prints before anything slow.import sys, platform, subprocess, timeSTART = time.time()print("=" * 58, flush=True)print(f"  notebook started  {time.strftime('%Y-%m-%d %H:%M:%S')}", flush=True)print(f"  python {sys.version.split()[0]} on {platform.platform()}", flush=True)try:    n = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],                       capture_output=True, text=True, timeout=20).stdout.strip()    print(f"  gpu: {n or 'none (fine for harvesting)'}", flush=True)except Exception:    print("  gpu: none (fine for harvesting)", flush=True)print(f"  internet: ", end="", flush=True)try:    import urllib.request    urllib.request.urlopen("https://pypi.org", timeout=15)    print("ON", flush=True)except Exception as e:    print(f"OFF or blocked -- {type(e).__name__}. "          "Settings > Internet > On (needs a phone-verified account).", flush=True)print("=" * 58, flush=True)

# Train YOLO26 on FRC fuel — KaggleKaggle over Colab for one reason: **Save & Run All** executes this headless,with no browser tab open. Colab's free tier kills the runtime when the tabcloses.## Before you run anythingIn the right-hand **Settings** panel:| setting | value | why || --- | --- | --- || Accelerator | **GPU T4 x2** (or P100) | otherwise it trains on CPU, silently || Internet | **On** | pip needs it; off by default, and enabling it needs a phone-verified account || Environment | Latest | ultralytics wants a current torch |Then **Add Input → Datasets → Your Datasets** and attach the dataset youuploaded (see the setup guide). It mounts read-only at `/kaggle/input/<name>/`.

## 1. Confirm the GPU is actually attachedIf this prints nothing, you left the accelerator off and the run will be ~20x slower than your Mac.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || print("NO GPU - fix Settings > Accelerator")

In [ ]:
!pip -q install "ultralytics>=8.4"import torch, ultralyticsprint("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())assert torch.cuda.is_available(), "No CUDA. Settings > Accelerator > GPU, then re-run."print("  pip install finished", flush=True)

## 2. Copy the dataset off the read-only mount**This is the Kaggle-specific trap.** `/kaggle/input` is read-only, andultralytics writes a `labels.cache` file next to the labels the first time itscans them. Training straight from `/kaggle/input` fails on that write.Copy to `/kaggle/working` (writable, and the part that persists into thecommitted version).

In [ ]:
import glob, shutil, tarfilefrom pathlib import Pathsrc = glob.glob('/kaggle/input/*/dataset.tgz') + glob.glob('/kaggle/input/*/dataset/')print("found in input:", src)work = Path('/kaggle/working/dataset')if work.exists():    shutil.rmtree(work)if src and src[0].endswith('.tgz'):    with tarfile.open(src[0]) as t:        t.extractall('/kaggle/working/')else:    shutil.copytree(src[0].rstrip('/'), work)print("train imgs:", len(list((work/'images'/'train').glob('*.jpg'))))print("train lbls:", len(list((work/'labels'/'train').glob('*.txt'))))print("val   imgs:", len(list((work/'images'/'val').glob('*.jpg'))))

### Rewrite dataset.yamlIt records an absolute path from your Mac.

In [ ]:
from pathlib import Pathy = Path('/kaggle/working/dataset/dataset.yaml')lines = [l for l in y.read_text().splitlines() if not l.startswith('path:')]y.write_text('path: /kaggle/working/dataset\n' + '\n'.join(lines) + '\n')print(y.read_text())

## 3. Train`batch=-1` is Ultralytics' AutoBatch: it probes the card and sizes the batch toroughly 60% of VRAM. Do not hardcode it. The P2 head at imgsz=1280 ismemory-hungry — a stride-4 feature map is sixteen times the area of stride-16 —and a fixed `batch=8` needs something like 24 GB, which OOMs a 16 GB T4.`imgsz` is the setting that actually matters for this dataset. Fuel is ~17x12px, so resolution is what makes it detectable at all; `yolo26s-p2` adds astride-4 head for the same reason.

In [ ]:
from ultralytics import YOLOmodel = YOLO('yolo26s-p2.yaml').load('yolo26s.pt')model.train(    data='/kaggle/working/dataset/dataset.yaml',    imgsz=1280, epochs=100, batch=-1,   # AutoBatch: sizes to ~60% of VRAM device=0,    project='/kaggle/working/runs', name='fuel26',    scale=0.25, mosaic=1.0, close_mosaic=15,    fliplr=0.5, flipud=0.0, degrees=0.0,    patience=30,          # stops once val mAP plateaus    workers=2,            # Kaggle gives ~4 vCPU; 8 workers thrash)

## 4. Keep the weights small enough to persist`/kaggle/working` is capped (about 20 GB) and the whole thing is saved into thecommitted version. The dataset copy is the bulky part and you do not need itback — delete it so the output is just the weights.

In [ ]:
import shutilfrom pathlib import Pathshutil.copy('/kaggle/working/runs/fuel26/weights/best.pt', '/kaggle/working/best.pt')shutil.copy('/kaggle/working/runs/fuel26/results.csv',     '/kaggle/working/results.csv')shutil.rmtree('/kaggle/working/dataset', ignore_errors=True)for p in sorted(Path('/kaggle/working').iterdir()):    print(p.name, f"{p.stat().st_size/1e6:.1f} MB" if p.is_file() else "(dir)")

## 5. Did it learn anything?

In [ ]:
import pandas as pddf = pd.read_csv('/kaggle/working/results.csv')cols = [c for c in df.columns if 'mAP' in c or 'precision' in c or 'recall' in c]print(df[['epoch'] + cols].tail(10).to_string(index=False))

## Running it unattended**Save Version → Save & Run All (Commit).** Close the tab. Kaggle runs it tocompletion in the background and the output files are attached to that versionwhen it finishes; download `best.pt` from the version's Output tab.Sessions cap around 12 hours. With `patience=30` this dataset finishes wellinside that.## Then, on your Mac```bashmkdir -p weights && mv ~/Downloads/best.pt weights/.venv-train/bin/yolo predict model=weights/best.pt source=data/frames/ imgsz=1280```